# README

- **Author**: RUEI-HSIANG, CHANG／張睿庠（B12303030；AI協助）
- **Created At**: 2026-11-08
- **Last Modified At**: 2026-11-08

---

## What does this file do?

依本次學生明確指令，在209人兩市版外另建210人「游智彬研究暫納入版」：合併210、首長11、議員199。
既有209人及其G5／G6編碼逐格保留；游智彬沿用上游G5=0，G6依已完成搜尋與既定三親等／職務範圍編0。
三立／鏡週刊全文記其外叔公游日正曾任鄉長／立委，依親等計算為四親等，不計入三親等範圍；外公軍人轉業文官依學生既定規則排除。
G6=0表示搜尋未發現符合範圍的親屬，不能寫成「完全沒有政治人物親戚」。
官方10/16資格不符、11/6暫時救濟駁回之紀錄保留；**研究暫納入不是官方認定合格**。
「審查中」在本研究註記為「資格爭議／待後續確認；研究暫納入」，避免誤述官方審查尚未作成結果。
不在這個準備Notebook擬合分群；新finding執行G5／G6三池與納入／排除敏感度，僅能評估本次分群，不能估計實際選舉結果影響。

---

## What does this file take?

以下每個精確檔案分別固定SHA，讀取前核對。原三池與資格稽核之文件為[taipei_kaohsiung原版文件](../processed/taipei_kaohsiung_20261108_documentation.ipynb)；
全國上游游智彬既有列為[clustering原版文件](../processed/clustering_20261108_documentation.ipynb)；
7份新增原始來源與決定為[本輪raw文件](../raw/you_zhibin_provisional_20261108_documentation.ipynb)。

| Input path | Expected SHA-256 |
| --- | --- |
| `data/processed/taipei_kaohsiung_all_20261108.csv` | `90959d176ac9cb34dcd264c52df9e2690ba70b0fc7cd483f58fac320c41d5899` |
| `data/processed/taipei_kaohsiung_mayor_20261108.csv` | `3b35f16286a7ea5d842d73f018ab4de5dc0900e4dd0bd1958da3c9ca9d4066e8` |
| `data/processed/taipei_kaohsiung_council_20261108.csv` | `6f4791a6347009636b91fb0bb97e47a77ac684fa7ac3c81e875d5f3b175ee644` |
| `data/processed/taipei_kaohsiung_eligibility_audit_20261108.csv` | `9f3868462b2f61d5c6ad7e16ff915516f1bb9ad950fb0a8b2d7775bc4ee8e830` |
| `data/processed/clustering_all_20261108.csv` | `2a70db4a868a80b35934d0437196cc8f189d27c6625e7a67b0bb6e753c0305bc` |
| `data/raw/you_zhibin_provisional_20261108/court_20261106_fulltext.txt` | `7c0377ed5e67966b3c2c23237c9822f27325eb84641a483ac298a46fcec640f9` |
| `data/raw/you_zhibin_provisional_20261108/family_mirror_fulltext.txt` | `c4febfeb8180ab084bf5d004b0faf0f2fbce3da15e0188353a6805ba173260a2` |
| `data/raw/you_zhibin_provisional_20261108/family_observer_fulltext.txt` | `0693acac19cfc935760450d55a27baa0956f7752e1f9d7cfe36da4b9b8a14e8e` |
| `data/raw/you_zhibin_provisional_20261108/kinship_law_fulltext.txt` | `76cba03314e23af641b31dec3f1e9380dcd2e8270a9da702231cd821bb46a3de` |
| `data/raw/you_zhibin_provisional_20261108/user_inclusion_decision.json` | `2c200f01586e9e9c9cc885841dca8aea62198242710609a6cdd3526503f6084f` |
| `data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json` | `baf49b670061b8cd8454718c452bd658be7f1da60b1b29b896351589a54302ea` |
| `data/raw/you_zhibin_provisional_20261108/source_manifest.json` | `410622864ebc202fbaf1044388a66102f4d74378e824369efd0d135d6859b75a` |

---

## What does this file output?

- `data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv`：210×51，analysis input；包含游智彬1位暫納入者。
- `data/processed/taipei_kaohsiung_yu_provisional_mayor_20261108.csv`：11×51，analysis input；人員與原首長池相同，新增研究狀態欄。
- `data/processed/taipei_kaohsiung_yu_provisional_council_20261108.csv`：199×51，analysis input。
- `data/processed/taipei_kaohsiung_yu_provisional_eligibility_audit_20261108.csv`：210列，保留原官方排除紀錄及新研究暫納入旗標，qualification／inclusion audit。
- `data/processed/taipei_kaohsiung_yu_provisional_manifest_20261108.json`：4份CSV之列欄數、SHA與回讀狀態。
- `data/processed/taipei_kaohsiung_yu_provisional_validation_20261108.json`：來源pins、唯一鍵、三池、原209人不變及資格分離等實際檢查。

各檔由本Notebook產生，逐檔文件：[processed文件](../processed/taipei_kaohsiung_yu_provisional_20261108_documentation.ipynb)。下游精確讀入上述三池；不覆寫舊209人、原始資格稽核或全國上游。

In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display,Markdown
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'templates').is_dir()),None)
assert ROOT is not None

In [2]:
PINS = {'data/processed/taipei_kaohsiung_all_20261108.csv': '90959d176ac9cb34dcd264c52df9e2690ba70b0fc7cd483f58fac320c41d5899', 'data/processed/taipei_kaohsiung_mayor_20261108.csv': '3b35f16286a7ea5d842d73f018ab4de5dc0900e4dd0bd1958da3c9ca9d4066e8', 'data/processed/taipei_kaohsiung_council_20261108.csv': '6f4791a6347009636b91fb0bb97e47a77ac684fa7ac3c81e875d5f3b175ee644', 'data/processed/taipei_kaohsiung_eligibility_audit_20261108.csv': '9f3868462b2f61d5c6ad7e16ff915516f1bb9ad950fb0a8b2d7775bc4ee8e830', 'data/processed/clustering_all_20261108.csv': '2a70db4a868a80b35934d0437196cc8f189d27c6625e7a67b0bb6e753c0305bc', 'data/raw/you_zhibin_provisional_20261108/court_20261106_fulltext.txt': '7c0377ed5e67966b3c2c23237c9822f27325eb84641a483ac298a46fcec640f9', 'data/raw/you_zhibin_provisional_20261108/family_mirror_fulltext.txt': 'c4febfeb8180ab084bf5d004b0faf0f2fbce3da15e0188353a6805ba173260a2', 'data/raw/you_zhibin_provisional_20261108/family_observer_fulltext.txt': '0693acac19cfc935760450d55a27baa0956f7752e1f9d7cfe36da4b9b8a14e8e', 'data/raw/you_zhibin_provisional_20261108/kinship_law_fulltext.txt': '76cba03314e23af641b31dec3f1e9380dcd2e8270a9da702231cd821bb46a3de', 'data/raw/you_zhibin_provisional_20261108/user_inclusion_decision.json': '2c200f01586e9e9c9cc885841dca8aea62198242710609a6cdd3526503f6084f', 'data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json': 'baf49b670061b8cd8454718c452bd658be7f1da60b1b29b896351589a54302ea', 'data/raw/you_zhibin_provisional_20261108/source_manifest.json': '410622864ebc202fbaf1044388a66102f4d74378e824369efd0d135d6859b75a'}
for rel,expected in PINS.items():
    assert (ROOT/rel).is_file() and hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()==expected,rel
raw=ROOT/'data/raw/you_zhibin_provisional_20261108'
decision=json.loads((raw/'user_inclusion_decision.json').read_text())
search=json.loads((raw/'G6_search_and_coding.json').read_text())
key=decision['candidate_key']
assert key==search['candidate_key'] and decision['not_an_official_approval']
assert search['G6_value']=='0' and len(search['queries'])>=2
assert {r.get('kinship_degree') for r in search['excluded_leads'] if 'kinship_degree' in r}=={4}
assert '外叔公游日正' in (raw/'family_mirror_fulltext.txt').read_text()
assert '公務員' in (raw/'family_observer_fulltext.txt').read_text()
assert '抗告駁回' in (raw/'court_20261106_fulltext.txt').read_text()
assert '旁系血親' in (raw/'kinship_law_fulltext.txt').read_text()
base={pool:pd.read_csv(ROOT/f'data/processed/taipei_kaohsiung_{pool}_20261108.csv',dtype=str,keep_default_na=False) for pool in ['all','mayor','council']}
source=pd.read_csv(ROOT/'data/processed/clustering_all_20261108.csv',dtype=str,keep_default_na=False)
old_audit=pd.read_csv(ROOT/'data/processed/taipei_kaohsiung_eligibility_audit_20261108.csv',dtype=str,keep_default_na=False)
assert (len(base['all']),len(base['mayor']),len(base['council']))==(209,11,198)
assert len(old_audit)==210 and key not in set(base['all'].candidate_key)
row=source[source.candidate_key.eq(key)].copy()
assert len(row)==1 and row.iloc[0].G5_first_time_any_elected_office=='0'
assert row.iloc[0].G6_political_family==''
print('12份來源SHA、原209人與新納入身分核對通過。')

12份來源SHA、原209人與新納入身分核對通過。


In [3]:
EXTRA=['research_inclusion_status_20261108','official_eligibility_note_20261108',
       'eligibility_provisional_flag_20261108','provisional_inclusion_source_20261108']
all_data=base['all'].copy()
all_data[EXTRA[0]]='baseline_included'
all_data[EXTRA[1]]='原209人版資格稽核保留；本輪未重審。'
all_data[EXTRA[2]]='0';all_data[EXTRA[3]]=''
yu={col:row.iloc[0].get(col,'') for col in all_data.columns}
yu['eligibility_status_20261108']='not_approved_20261016_disputed_provisional_research_include'
yu['eligibility_source_20261108']='https://web.cec.gov.tw/central/article/65212; https://www.judicial.gov.tw/tw/cp-1888-1645308-4e8cf-1.html'
yu['TK_completion_checked_date']='2026-11-08'
yu['G6_political_family']=search['G6_value']
yu['G6_label']=search['G6_label']
yu['G6_evidence']=search['evidence']
yu['G6_source']='; '.join(s['source_url'] for s in search['full_text_sources'])+'; data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json'
yu['G6_evidence_tier']=search['evidence_tier']
yu['G6_search_status_20261108']='searched_no_qualifying_evidence'
yu['G6_review_note_tk_20261108']=search['evidence']+'；'+search['limitations']
yu['checked_date']='2026-11-08'
yu[EXTRA[0]]='provisional_included_eligibility_disputed'
yu[EXTRA[1]]=decision['official_status_preserved']
yu[EXTRA[2]]='1';yu[EXTRA[3]]='data/raw/you_zhibin_provisional_20261108/user_inclusion_decision.json'
all_data=pd.concat([all_data,pd.DataFrame([yu],columns=all_data.columns)],ignore_index=True)
pools={'all':all_data,'mayor':all_data[all_data.office_type.eq('直轄市長')].copy(),
       'council':all_data[all_data.office_type.eq('直轄市議員')].copy()}
assert all_data.candidate_key.is_unique and len(all_data.columns)==51
assert (len(pools['all']),len(pools['mayor']),len(pools['council']))==(210,11,199)
features=['G5_first_time_any_elected_office','G6_political_family']
for pool,d in pools.items():
    assert d[features].isin(['0','1']).all().all()
    pd.testing.assert_frame_equal(d[d.candidate_key.isin(base[pool].candidate_key)][base[pool].columns].reset_index(drop=True),base[pool].reset_index(drop=True))
    assert d.eligibility_provisional_flag_20261108.eq('1').sum()==(0 if pool=='mayor' else 1)
assert set(pools['mayor'].candidate_key).isdisjoint(pools['council'].candidate_key)
assert set(pools['mayor'].candidate_key)|set(pools['council'].candidate_key)==set(pools['all'].candidate_key)
audit=old_audit.rename(columns={'retained_in_pool':'baseline_retained_in_pool','reason':'baseline_exclusion_reason'}).copy()
audit['research_retained_in_pool']='1'
audit['research_inclusion_status_20261108']=audit.candidate_key.map(all_data.set_index('candidate_key').research_inclusion_status_20261108)
audit['eligibility_provisional_flag_20261108']=audit.candidate_key.map(all_data.set_index('candidate_key').eligibility_provisional_flag_20261108)
audit['official_eligibility_note_20261108']=audit.candidate_key.map(all_data.set_index('candidate_key').official_eligibility_note_20261108)
audit['research_decision_source']=audit.candidate_key.map(all_data.set_index('candidate_key').provisional_inclusion_source_20261108)
assert audit[audit.candidate_key.eq(key)].baseline_retained_in_pool.eq('0').all()
assert audit[audit.candidate_key.eq(key)].research_retained_in_pool.eq('1').all()
assert set(audit.candidate_key)==set(all_data.candidate_key)
display(Markdown('**表1：研究暫納入後三池（非互斥）；距離只使用G5／G6。**'))
display(pd.DataFrame([{'樣本池':p,'人數':len(d),'G5缺值':int(d[features[0]].eq('').sum()),'G6缺值':int(d[features[1]].eq('').sum()),'資格爭議暫納入':int(d.eligibility_provisional_flag_20261108.eq('1').sum())} for p,d in pools.items()]))
display(all_data[all_data.candidate_key.eq(key)][['candidate_name',*features,*EXTRA]])

**表1：研究暫納入後三池（非互斥）；距離只使用G5／G6。**

,樣本池,人數,G5缺值,G6缺值,資格爭議暫納入
0,all,210,0,0,1
1,mayor,11,0,0,0
2,council,199,0,0,1


,candidate_name,G5_first_time_any_elected_office,G6_political_family,research_inclusion_status_20261108,official_eligibility_note_20261108,eligibility_provisional_flag_20261108,provisional_inclusion_source_20261108
209,游智彬,0,0,provisional_included_eligibility_disputed,10/16中選會審定資格不符；11/6最高行政法院駁回暫時救濟抗告，本案法律爭議仍待訴訟釐清。,1,data/raw/you_zhibin_provisional_20261108/user_...


In [4]:
OUT=ROOT/'data/processed'
PREFIX='taipei_kaohsiung_yu_provisional_'
outputs={f'{PREFIX}{pool}_20261108.csv':d for pool,d in pools.items()}
outputs[f'{PREFIX}eligibility_audit_20261108.csv']=audit
manifest=[]
for name,d in outputs.items():
    p=OUT/name
    d=d.fillna('').astype(str)
    d.to_csv(p,index=False,encoding='utf-8-sig')
    reread=pd.read_csv(p,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(reread,d.reset_index(drop=True))
    manifest.append({'path':str(p.relative_to(ROOT)),'rows':len(reread),'columns':list(reread),
                     'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
                     'verification':'CSV reopened; exact schema, row order and all cells verified'})
for rel,expected in PINS.items():assert hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()==expected
(OUT/f'{PREFIX}manifest_20261108.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n')
validation={'date':'2026-11-08','input_bindings':PINS,'newly_included_key':key,
 'pools':{p:len(d) for p,d in pools.items()},'original_209_rows_all_cells_unchanged':True,
 'baseline_eligibility_exclusion_preserved':True,'official_approval_claimed':False,
 'research_provisional_inclusion':True,'G5_G6_missing':0,'G5':'0','G6':'0',
 'G6_search_record':'data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json',
 'three_pool_partition':True,'saved_csv_count':4,'csv_readback':True,'input_hashes_unchanged':True,
 'limitation':'G6=0 means no qualifying spouse/within-three-degree political relative found in recorded search; four-degree political kin excluded by existing rule. Research inclusion does not certify eligibility or predict election outcomes.'}
(OUT/f'{PREFIX}validation_20261108.json').write_text(json.dumps(validation,ensure_ascii=False,indent=2)+'\n')
assert json.loads((OUT/f'{PREFIX}manifest_20261108.json').read_text())==manifest
assert json.loads((OUT/f'{PREFIX}validation_20261108.json').read_text())==validation
print('PASS：4份CSV回讀，210=11+199，原209人逐格不變、官方紀錄保留與研究暫納入分離。')
display(pd.DataFrame(manifest)[['path','rows','sha256']])

PASS：4份CSV回讀，210=11+199，原209人逐格不變、官方紀錄保留與研究暫納入分離。


,path,rows,sha256
0,data/processed/taipei_kaohsiung_yu_provisional...,210,35e3b85e407d4eb91e5c96043d06a407d21e070a592003...
1,data/processed/taipei_kaohsiung_yu_provisional...,11,d8c448a6be69af7e6a291a376d0de31157c0267f7f4180...
2,data/processed/taipei_kaohsiung_yu_provisional...,199,a182db22545ca308286970e49c954d3a903af3afc7d6cd...
3,data/processed/taipei_kaohsiung_yu_provisional...,210,925c64943b010d755685327877634900692680a49f36ea...
